# Study 01 - Plain Python Foundations

This walkthrough starts with the same double-integrator idea used in the original `notebooks/plane_code/01_open_loop.ipynb` teaching material. The goal is to see the model, input sequence, feedback law, simulation loop, and plots directly in plain NumPy.


In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

STUDY_DIR = Path('studies/study_01_plain_python_foundations')
OUTPUT_DIR = STUDY_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Model

The state is

$$x_k = [p_k, v_k]^T$$

and the input is acceleration. With sample time $T_s$, the discrete model is

$$x_{k+1} = A x_k + B u_k.$$


In [ ]:
dt = 0.1
A = np.array([[1.0, dt], [0.0, 1.0]])
B = np.array([[0.5 * dt**2], [dt]])

x0 = np.array([1.0, 0.0])
steps = 80
u_min, u_max = -1.0, 1.0

print('A =\n', A)
print('B =\n', B)


## Open-Loop Simulation

First choose the complete input sequence before the simulation starts. The optimizer does not exist yet; the point is only to understand propagation.


In [ ]:
def simulate_linear(A, B, x0, U):
    X = np.zeros((len(U) + 1, len(x0)))
    X[0] = x0
    for k, u in enumerate(U):
        X[k + 1] = A @ X[k] + B[:, 0] * u
    return X

U_open = np.zeros(steps)
X_open = simulate_linear(A, B, x0, U_open)


## Feedback

Feedback computes the input from the current state. A first controller is just hand-tuned state feedback,

$$u_k = -K x_k.$$


In [ ]:
K = np.array([0.6, 1.1])
# SOLUTION_START
K = np.array([1.2, 1.8])
# SOLUTION_END

def simulate_feedback(A, B, x0, K, steps, u_min=None, u_max=None):
    X = np.zeros((steps + 1, len(x0)))
    U = np.zeros(steps)
    X[0] = x0
    for k in range(steps):
        u = -float(K @ X[k])
        if u_min is not None and u_max is not None:
            u = float(np.clip(u, u_min, u_max))
        U[k] = u
        X[k + 1] = A @ X[k] + B[:, 0] * u
    return X, U

X_fb, U_fb = simulate_feedback(A, B, x0, K, steps, u_min, u_max)


## Plots And Interpretation

The plot shows the difference between a preselected input sequence and a feedback rule that reacts to the state.


In [ ]:
t = np.arange(steps + 1) * dt
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].plot(t, X_open[:, 0], label='open loop')
axes[0].plot(t, X_fb[:, 0], label='feedback')
axes[1].plot(t, X_open[:, 1], label='open loop')
axes[1].plot(t, X_fb[:, 1], label='feedback')
axes[2].step(t[:-1], U_open, where='post', label='open loop')
axes[2].step(t[:-1], U_fb, where='post', label='feedback')
axes[2].axhline(u_min, color='k', linestyle='--', linewidth=0.8)
axes[2].axhline(u_max, color='k', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('position')
axes[1].set_ylabel('velocity')
axes[2].set_ylabel('input')
axes[2].set_xlabel('time [s]')
for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend(loc='best')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'open_loop_vs_feedback.png', dpi=150)
plt.show()

print('final feedback state:', X_fb[-1])
print('max abs input:', np.max(np.abs(U_fb)))


## Student Questions

- What is decided before simulation in open loop?
- What measurement does feedback use at every step?
- What happens if the input clipping is removed?
